# Task 0.2: Exploratory Data Analysis & Data Quality Validation

**Spec Reference:** Section 7.2, Lines 2826-2844

This notebook validates data quality assumptions from the specification.

In [1]:
# Cell 1: Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

/home/dacthinh/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Cell 2: Load January 2024 data
data_path = Path('../data/raw/yellow_tripdata_2024-01.parquet')
df = pd.read_parquet(data_path)

print(f"Records: {len(df):,}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Columns: {len(df.columns)}")

Records: 2,964,624
Memory: 0.56 GB
Columns: 19


In [3]:
# Cell 3: Null rates (Spec Lines 2135-2140)
null_rates = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Null Rates (%):")
print(null_rates[null_rates > 0])

passenger_null_rate = null_rates.get('passenger_count', 0)
print(f"\npassenger_count null rate: {passenger_null_rate:.2f}%")
print(f"Expected: 4.0-6.0% (spec validation)")

if 4.0 <= passenger_null_rate <= 6.0:
    print("✅ Within spec range")
else:
    print("⚠️ Outside spec - update design doc")

Null Rates (%):
Airport_fee             4.727817
congestion_surcharge    4.727817
passenger_count         4.727817
RatecodeID              4.727817
store_and_fwd_flag      4.727817
dtype: float64

passenger_count null rate: 4.73%
Expected: 4.0-6.0% (spec validation)
✅ Within spec range


In [4]:
# Cell 4: Figure 1 - Temporal Trends
df['pickup_dt'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['hour'] = df['pickup_dt'].dt.hour

plt.figure(figsize=(14, 6))
hourly_volume = df.groupby('hour').size()
hourly_volume.plot(marker='o', linewidth=2, color='steelblue')
plt.axvspan(7, 9, alpha=0.2, color='red', label='Morning Rush')
plt.axvspan(17, 19, alpha=0.2, color='blue', label='Evening Rush')
plt.title('Phase 0: Temporal Trends - Hourly Pattern', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Trip Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/figures/phase0_temporal_trends.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: phase0_temporal_trends.png")

✅ Saved: phase0_temporal_trends.png


In [5]:
# Cell 5: Figure 2 - Quality Evolution
plt.figure(figsize=(14, 6))
# Load all months to show evolution
monthly_null_rates = []
for month in range(1, 13):
    df_month = pd.read_parquet(f'../data/raw/yellow_tripdata_2024-{month:02d}.parquet')
    null_rate = df_month.isnull().sum().sum() / (len(df_month) * len(df_month.columns)) * 100
    monthly_null_rates.append(null_rate)

plt.plot(range(1, 13), monthly_null_rates, marker='o', linewidth=2, color='orange')
plt.title('Phase 0: Data Quality Evolution (2024)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Null Rate (%)')
plt.axhline(y=5.0, color='red', linestyle='--', label='Threshold (5%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/figures/phase0_quality_evolution.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: phase0_quality_evolution.png")

✅ Saved: phase0_quality_evolution.png


In [6]:
# Cell 6: Figure 3 - Spatial Heatmap (PU × DO)
plt.figure(figsize=(14, 10))
# Sample 10K records for heatmap (too large otherwise)
df_sample = df.sample(10000, random_state=42)
heatmap_data = df_sample.groupby(['PULocationID', 'DOLocationID']).size().unstack(fill_value=0)

# Top 30 zones only (readable heatmap)
top_zones = df['PULocationID'].value_counts().head(30).index
heatmap_filtered = heatmap_data.loc[top_zones, top_zones]

sns.heatmap(heatmap_filtered, cmap='YlOrRd', cbar_kws={'label': 'Trip Count'}, linewidths=0.5)
plt.title('Phase 0: Spatial Distribution Heatmap (Top 30 Zones)', fontsize=14, fontweight='bold')
plt.xlabel('Dropoff Location ID')
plt.ylabel('Pickup Location ID')
plt.tight_layout()
plt.savefig('../docs/figures/phase0_spatial_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: phase0_spatial_heatmap.png")

# Note: Figure 4 (phase0_synthetic_distribution.png) will be generated in Task 0.6

✅ Saved: phase0_spatial_heatmap.png


In [7]:
# Cell 7: Outliers (Spec Lines 2165-2180)
outliers = {
    'negative_fare': (df['fare_amount'] < 0).sum(),
    'negative_distance': (df['trip_distance'] < 0).sum(),
    'zero_distance': (df['trip_distance'] == 0).sum(),
    'extreme_distance': (df['trip_distance'] > 100).sum(),
    'extreme_fare': (df['fare_amount'] > 500).sum(),
    'passenger_zero': (df['passenger_count'] == 0).sum(),
    'passenger_high': (df['passenger_count'] > 6).sum(),
}

print("Outlier Counts:")
for key, count in outliers.items():
    pct = count / len(df) * 100
    print(f"{key:20s}: {count:8,} ({pct:5.2f}%)")

# Geographic violations
valid_zones = set(range(1, 264))
invalid_pu = df[~df['PULocationID'].isin(valid_zones)]
invalid_do = df[~df['DOLocationID'].isin(valid_zones)]

print(f"\nInvalid PULocationID: {len(invalid_pu):,}")
print(f"Invalid DOLocationID: {len(invalid_do):,}")

Outlier Counts:
negative_fare       :   37,448 ( 1.26%)
negative_distance   :        0 ( 0.00%)
zero_distance       :   60,371 ( 2.04%)
extreme_distance    :       59 ( 0.00%)
extreme_fare        :       46 ( 0.00%)
passenger_zero      :   31,465 ( 1.06%)
passenger_high      :       60 ( 0.00%)

Invalid PULocationID: 12,018
Invalid DOLocationID: 28,083


In [8]:
# Cell 8: Business rules (Layer 2 preview)
violations = {
    'negative_fare': df['fare_amount'] <= 0,
    'zero_dist_with_fare': (df['trip_distance'] == 0) & (df['fare_amount'] > 0),
    'excess_passengers': df['passenger_count'] > 6,
    'invalid_payment': ~df['payment_type'].isin([1, 2, 3, 4, 5, 6]),
}

total_viol = sum(v.sum() for v in violations.values())
viol_rate = total_viol / len(df) * 100

print("Business Rule Violations:")
for rule, mask in violations.items():
    count = mask.sum()
    pct = count / len(df) * 100
    print(f"{rule:25s}: {count:8,} ({pct:5.2f}%)")

print(f"\nTotal violation rate: {viol_rate:.2f}%")
print(f"Spec claim: ~13.49% filtered by Layer 1+2")

Business Rule Violations:
negative_fare            :   38,341 ( 1.29%)
zero_dist_with_fare      :   56,569 ( 1.91%)
excess_passengers        :       60 ( 0.00%)
invalid_payment          :  140,162 ( 4.73%)

Total violation rate: 7.93%
Spec claim: ~13.49% filtered by Layer 1+2


In [9]:
# Cell 9: Generate markdown report
# First compute spatial and temporal stats
coverage_pct = df['PULocationID'].nunique() / 263 * 100
hourly = df.groupby('hour').size()
df['day'] = df['pickup_dt'].dt.date
daily = df.groupby('day').size()

findings = f"""# EDA Findings - January 2024

## Dataset
- Records: {len(df):,}
- Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB
- Columns: {len(df.columns)}

## Null Rates (Top 5)
{null_rates.head().to_frame('Null Rate (%)').to_markdown()}

## Spatial
- Zones: {df['PULocationID'].nunique()}/263
- Coverage: {coverage_pct:.1f}%

## Temporal
- Peak hour: {hourly.idxmax()}:00
- Busiest day: {daily.idxmax()}

## Outliers
{pd.Series(outliers).to_frame('count').to_markdown()}

## Business Rules
- Violation rate: {viol_rate:.2f}%

## Recommendations
1. Baseline sanitization (Task 0.5) CRITICAL
2. Neighborhood mapping (Task 0.4)
3. Synthetic anomalies (Task 0.6)
"""

Path('../docs').mkdir(exist_ok=True)
Path('../docs/figures').mkdir(exist_ok=True)
Path('../docs/eda_findings.md').write_text(findings)
print("✅ Saved to docs/eda_findings.md")

✅ Saved to docs/eda_findings.md
